# Task 4 · Reconstruct the Flight (20 marks)

Connect your trackers to the provided pipeline and recover the lander's
velocity and 3D trajectory, then find out how the quality of your optical flow
decides the result. You write only the four short functions of Task 4.1; the
camera-motion fit (`provided/task4/motion_model.py`), the flight code
(`provided/task4/trajectory.py`) and the plots are provided.

The camera points straight down at flat ground. As the lander moves, the
tracked ground features between two consecutive frames undergo a similarity:
they scale, rotate and translate,

$$
\mathbf{x}_{k+1} = s_k R(\theta_k)\,\mathbf{x}_k + \mathbf{t}_k,
\qquad s_k = \frac{Z_k}{Z_{k+1}}.
$$

The provided fit estimates $s_k$, $\theta_k$ and $\mathbf{t}_k$ by robust least
squares, and the flight code turns them into altitude, position, velocity and
the 3D trajectory, in metres and seconds. The world frame is fixed to frame
270: $X$ to the right, $Y$ toward its top (image $y$ points down), $Z$ the
altitude (up).

**How this notebook works**

* Write your code only in the cells whose first line starts with `# YOUR CODE`.
  Only those cells are graded, and they are read from the **saved** notebook.
  Add your own cells anywhere for experiments or notes: they are never graded.
  A helper function you want graded with your code goes in a `# YOUR CODE` cell
  (a new cell whose first line is `# YOUR CODE` works too).
* Run the `check(...)` cell after your code: it tests your functions and prints
  PASS or FAIL for each test.
* Answer each question in the cell right below it, in one or two sentences,
  replacing *TODO*.
* Save often (Ctrl+S). Before you submit, run everything from a clean start:
  restart the kernel and run all cells, or run `python run_notebooks.py`.

In [ ]:
# PROVIDED: run this cell first (save task3.ipynb before)
%matplotlib inline
import numpy as np
from provided.notebook import check, use_my_functions
from provided.sift import sift_matches, track_with_sift
from provided.task4.compare_trackers import compare
from provided.task4.run import read_motion, reconstruct_flight

use_my_functions(3)               # your Task 1 to 3 functions, from the saved notebooks

## 4.1 Connect your trackers to the pipeline (4 marks)

Fill in the three adapters `one_step`, `iterative` and `sift_guided`, which
call your Task 3 `track_points`, and `track_pair`, which selects points
(Task 1.3), runs one tracker and keeps the valid tracks. A tracker is a
function `(image0, image1, points) -> (tracked, valid)`.

In [ ]:
# YOUR CODE: Task 4.1

# the settings every tracker uses (keep them for the reference numbers)
MAX_POINTS = 200
FEATURE_RADIUS = 3
MIN_DISTANCE = 10
TRACK_RADIUS = 7
ITERATIONS = 10

def one_step(image0, image1, points):
    """One Lucas-Kanade update per point (the Task 2 method)."""
    return track_points(image0, image1, points, TRACK_RADIUS, max_iterations=1)


def iterative(image0, image1, points):
    """Iterative Lucas-Kanade per point (Task 3)."""
    return track_points(image0, image1, points, TRACK_RADIUS, max_iterations=ITERATIONS)


def sift_guided(image0, image1, points):
    """Iterative Lucas-Kanade started from SIFT guesses (Task 3.3)."""
    matches = sift_matches(image0, image1)
    return track_with_sift(track_points, image0, image1, points, matches, TRACK_RADIUS,
                           max_iterations=ITERATIONS)


def track_pair(image0, image1, tracker):
    """Select points in image0 and track them into image1 with `tracker`.

    Returns (points0, points1), keeping only the valid tracks.
    """
    points = select_features(image0, MAX_POINTS, FEATURE_RADIUS, MIN_DISTANCE)
    tracked, valid = tracker(image0, image1, points)
    valid = np.asarray(valid, dtype=bool)
    return points[valid], np.asarray(tracked)[valid]

In [ ]:
check("4.1")

trackers = {"one-step": one_step, "iterative": iterative, "sift-guided": sift_guided}

**(a)** Why must `track_pair` drop the failed tracks before the motion fit?

A failed track has a wrong position (often just where it started), and the fit would treat it as real and get the zoom, rotation and shift wrong. The fit can ignore a few bad points, but failures can be most of them (60 % on 409 to 410), so they must be removed first.

## 4.2 Read the motion from the images (4 marks)

The provided cell tracks all 143 frame pairs with your SIFT-guided tracker
(a few minutes), fits the camera motion of every pair and plots the zoom,
rotation and shift.

In [ ]:
motion = read_motion(track_pair, trackers["sift-guided"])

**(a)** Why is the zoom $s_k = Z_k / Z_{k+1}$? Use the pinhole model.

In the pinhole model a ground point at distance X from the centre shows up at x = f X / Z. In the next frame it is at x' = f X / Z', so x' / x = Z / Z' = s. As the lander goes down, Z' gets smaller and the image zooms in (s > 1).

**(b)** What are the phases of the descent, and at about which frames do they change?

Three phases: (1) **main descent**, frames 270 to about 365: zoom 1.02 to 1.045 per frame and large shifts, so the lander drops fast and moves forward; (2) **hover**, about 365 to 383: zoom about 1.000, with a short sideways move and a 1 to 2° turn around 370 to 374; (3) **final drop**, about 384 to 413: zoom rises above 1.15 per frame with small shifts.

**(c)** Which pattern in these plots is not caused by the lander, and how can you tell?

The spikes every 5th pair (274, 279, 284, ...) are not caused by the lander. There the zoom and shift are almost exactly double the neighbouring values (for example 1.057 at 304 against 1.028), and a real move would not repeat on such a regular beat. It means a camera frame was skipped, so that pair covers 2 seconds, which is why the flight lasts 168 s for 143 pairs.

## 4.3 Reconstruct the flight (4 marks)

The provided cell turns the motion into the altitude, the velocity
$(v_X, v_Y, v_Z)$, the time to touchdown $\tau = Z / (-v_Z)$, the 3D trajectory
and the ground track (seen from above, $X$ to the right and $Y$ up).

In [ ]:
flight, summary = reconstruct_flight(motion)

**Table.** Fill in the numbers printed above (also in `outputs/task4/summary.json`).

| Quantity | Value |
|----------|-------|
| Start and end altitude Z | 3666.6 m to 9.1 m |
| Duration | 168 s |
| Largest descent speed | 68.4 m/s |
| Horizontal drift | 2280 m |

**(a)** Which result do you trust least, and why?

The start altitude (3667 m) and the horizontal drift (2280 m). We only know the true altitude at frame 381, so to get back to frame 270 we chain 111 zoom values, and small errors add up; the drift depends on that altitude too. The last two pairs (411, 412) also have big errors (12 to 16 px).

## 4.4 Compare the three trajectories (5 marks)

The provided cell runs the whole pipeline once per tracker on the same points
and draws the three ground tracks together (several minutes the first time;
the tracks are saved and reused while your code stays the same).

In [ ]:
table = compare(trackers, track_pair)

**Table.** Fill in the numbers printed above (also in `outputs/task4/compare/summary.csv`).

| Tracker | End altitude | Horizontal drift | Median residual |
|---------|--------------|------------------|-----------------|
| one-step | 99.0 m | 1.3 m | 0.23 px |
| iterative | 92.9 m | 8.8 m | 1.79 px |
| SIFT-guided | 9.1 m | 2280 m | 0.35 px |

**(a)** Which trajectory do you trust, and which Task 3 result supports it?

SIFT-guided, since it is the only one where the lander actually comes down to near the ground; the other two say it stayed at about 100 m. Task 3.4 backs this up: on 300 to 301 only SIFT-guided found the real ~10 px motion.

**(b)** Does a low median residual mean a tracker is right? Use your numbers.

No. One step has the lowest residual (0.23 px, lower than SIFT-guided at 0.35 px), but its trajectory is clearly wrong. It barely moves the points, and almost no motion is easy to fit, so a low residual only means the tracks agree with each other, not that they are correct.

Optional, not graded: the next cell writes
`outputs/task4/trajectory_animation.html`, the three flights as an interactive
3D animation to open in a web browser.

In [ ]:
from provided.task4.animate_trajectory import main as animate
animate([])

## 4.5 Where does uncertainty come from? (3 marks)

**(a)** Why does a small error in each zoom grow along the descent?

Each altitude is worked out from the previous one using the zoom, so every small zoom error gets multiplied into all the later steps. Frame 270 is 111 steps away from the anchor at 381, so a 0.1 % error per step turns into about 11 % at the start.

**(b)** What changes if the anchor altitude is 10 % too high? And if the focal length is?

If the anchor altitude is 10 % too high, every altitude, position, drift and speed comes out 10 % too high, but the time to touchdown stays the same. If the focal length is 10 % too high, altitudes and descent speed don't change, but sideways distances and drift come out about 10 % too small.

# Final Mission Report · What Did We Learn? (bonus, 5 marks)

One or two sentences each, with one piece of evidence from your results.

**Q1.** Where in the descent does "all pixels in a window move together" fail most, and what in your results shows it?

Near touchdown, where the image zooms 1.15 to 1.25 times per frame and each window gets stretched instead of just shifted. On 409 to 410 SIFT-guided lost 60 % of its tracks, and the fit error jumps to 0.94 px at 409 and 12 to 16 px at 411 and 412.

**Q2.** Would a pyramidal (coarse-to-fine) KLT tracker fix 300 to 301? Would it fix 409 to 410? Why?

For 300 to 301, yes: it is mostly a 10 px shift, and shrinking the image 2 or 3 times cuts it to about 1 to 2.5 px, which iterative can handle. For 409 to 410, probably not: the motion is mostly zoom, and shrinking the image does not remove the stretching, so the windows still don't match and points still leave the frame.

**Q3.** If you ignored the pattern from Task 4.2, which Task 4.3 results would be wrong, and by how much?

Times and speeds would be wrong, but altitude and positions would not. The flight would last 143 s instead of 168 s (about 15 % short), and on the 25 skipped frame pairs the speed would come out about twice as high.

**Q4.** Where does the "flat ground, camera straight down" model fail, and what error does it cause?

It fails at the very end, when the lander is only 10 to 20 m up. Rocks and crater rims are no longer tiny next to that height, so points at different heights move differently, and one simple fit can't match them. You can see this in the 12 to 16 px errors at 411 and 412, and the flight ends at 9.1 m instead of 0.

**Q5.** Name one independent measurement that would check your trajectory, and what it would check.

The lander's radio tracking data (or its laser altimeter) gives a separate altitude over time. Comparing with it would check my altitudes, especially the 3667 m start I trust least, and the descent speeds.

# References

List every source you used, including the Chang'e-5 landing camera data (level 2A, https://moon.bao.ac.cn).

1. Chang'e-5 landing camera images, level 2A, frames 270 to 413, https://moon.bao.ac.cn
2. Altitude anchor at frame 381: Chang'e-5 descent trajectory from radio tracking, *Remote Sensing* 13, 4837 (2021).
3. ES666 Assignment 3 handout and provided code.
4. B. D. Lucas and T. Kanade, "An iterative image registration technique with an application to stereo vision", IJCAI 1981.
5. J. Shi and C. Tomasi, "Good features to track", CVPR 1994.
6. D. G. Lowe, "Distinctive image features from scale-invariant keypoints", IJCV 2004.